# nitid D-FINE Tutorial

This notebook walks through the basic nitid workflow:

1. Set up the project environment
2. Download and wrap an official D-FINE checkpoint
3. Run prediction on a real image
4. Inspect and save results
5. Export the model
6. See how fine-tuning would be launched

This notebook is written to run locally from the nitid repository. Comments are included for the small changes needed when running it later in Google Colab.

## 0. Local vs Colab setup

**Local use:** run this notebook from the repository root after installing dependencies.

**Google Colab:** running this notebook in Colab will automatically install nitid from GitHub in the cell below.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Detected Google Colab environment. Installing nitid...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "nitid @ git+https://github.com/Nitidio/nitid.git",
        ],
        check=True,
    )
    REPO_ROOT = Path("/content")
else:

    def find_repo_root(start: Path) -> Path:
        for candidate in [start, *start.parents]:
            if all((candidate / p).exists() for p in ["dfine", "tools", "configs"]):
                return candidate
        return start

    REPO_ROOT = find_repo_root(Path.cwd().resolve())

os.chdir(REPO_ROOT)
print(f"Running in Colab: {IN_COLAB}")
print(f"Repository root: {REPO_ROOT}")
print(f"Working directory: {Path.cwd()}")

## 1. Check the package imports

The public API is one entry point: import `NITID`, load a checkpoint, then call `predict()`, `train()`, `val()`, or `export()`.

In [ ]:
from nitid import NITID

print("Imported NITID successfully")

## 2. Load the model

nitid accepts public model identifiers such as `model1s` and downloads the official self-contained checkpoint automatically upon first use:


In [ ]:
model = NITID("model1s", task="detect")
print(f"Loaded model: {model.nitid_model}")

## 3. Download a test image

The image is downloaded at runtime so the repository does not need to store example image files.

In [ ]:
from urllib.request import urlretrieve

outputs_dir = REPO_ROOT / "outputs"
outputs_dir.mkdir(exist_ok=True)

image_url = "https://ultralytics.com/images/bus.jpg"
image_path = outputs_dir / "bus.jpg"

if not image_path.exists():
    print(f"Downloading {image_url}")
    urlretrieve(image_url, image_path)

print(f"Image ready: {image_path}")

## 4. Run prediction with Python

This is the main user-facing workflow.

In [ ]:
results = model.predict(str(image_path), conf=0.5, verbose=False)

result = results[0]
detections = result.to_json()
print(result)
print("Detections:", len(result))
print(detections)

## 5. Save and view the annotated image

`Results.save()` writes an annotated image with boxes and labels.

In [ ]:
try:
    from IPython.display import Image, display
except ImportError:
    Image = None
    display = None

annotated_path = outputs_dir / "bus_result.jpg"
result.save(str(annotated_path))
print(f"Saved annotated image to: {annotated_path}")

if Image is not None and display is not None:
    display(Image(filename=str(annotated_path)))
else:
    print("Install IPython or open the saved file to view the annotated image.")

## 6. Run prediction with the CLI

The model can also be run from the command line.

Inside a notebook, we invoke the CLI with Python:

```bash
nitid predict model=model1s source=outputs/bus.jpg conf=0.5
```


In [ ]:
cmd = [
    sys.executable,
    "-m",
    "nitid.cli",
    "predict",
    "model=model1s",
    f"source={image_path}",
    "conf=0.5",
]
print("Running:", " ".join(map(str, cmd)))
completed = subprocess.run(
    [str(x) for x in cmd],
    text=True,
    capture_output=True,
)

if completed.stdout:
    print(completed.stdout)
if completed.stderr:
    print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError(f"CLI prediction failed with exit code {completed.returncode}")

## 7. Export the model

For a quick local smoke test, this notebook exports to TorchScript. ONNX export is also supported, but can take longer and requires ONNX dependencies.

The exporter writes to the current working directory, so this cell temporarily switches into `outputs/` before exporting.

In [ ]:
exported_path = model.export(
    format="torchscript",
    imgsz=640,
    output=outputs_dir / "model1s.torchscript",
    exist_ok=True,
    verbose=False,
)
print(f"Exported model: {exported_path}")
print(f"Size: {exported_path.stat().st_size / 1e6:.1f} MB")

## 8. Fine-tuning workflow

Fine-tuning requires a COCO-format dataset and the `train` extra.

Local install:

```bash
uv sync --extra train
```

Future Colab install:

```python
!pip install -e ".[train]"
```

A real fine-tuning command looks like this:

```python
model.train(
    data="configs/datasets/example_custom.yml",
    epochs=10,
    batch=4,
    device="cuda",
)
```

This notebook does **not** run training by default because CPU training is slow and a real dataset path is required. Set `RUN_TRAINING = True` only after updating `DATA_YAML` to point at your dataset.

In [ ]:
RUN_TRAINING = False
DATA_YAML = "configs/datasets/example_custom.yml"

if RUN_TRAINING:
    metrics = model.train(
        data=DATA_YAML,
        epochs=1,
        batch=2,
        device="cpu",
        verbose=True,
    )
    print(metrics)
else:
    print("Skipping training. Set RUN_TRAINING=True and DATA_YAML to a real dataset to run it.")

## 9. What files were created?

Generated files are intentionally written to ignored folders:

```text
models/   # downloaded wrapped checkpoints
outputs/  # images and exported models
```

These should stay local and should not be committed to Git.

In [ ]:
for path in [image_path, annotated_path, exported_path]:
    if path.exists():
        print(path.name, f"{path.stat().st_size / 1e6:.1f} MB")